## Environment Setup

In [1]:
import os
import json
import xpd_tools
from tkinter.constants import N
import pandas as pd
import numpy as np

## AGENT - Initialization

### Initialize the Agent and set the logical paths and globals of our system.

For Bluesky:
- Queueserver toggle
- Tiled profile

Other:
- HTTP API and Server uri's
- ZMQ consumer addresses

Evaluation methods:
- xray, uvvis, or xray-uvis


In [2]:
from xpd_tools.optimization.agent import BuildAgent

# Queueserver connection -- unused on this build_local() path, kept here
# only because HTTP_SERVER_URI/HTTP_API_KEY/ZMQ_CONSUMER_ADDR are read from
# the environment below regardless.
HTTP_SERVER_URI = os.environ.get(
    "QSERVER_HTTP_URI", 
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov"
    )
HTTP_API_KEY = os.environ.get(
    "QSERVER_HTTP_SERVER_API_KEY", 
    ""
    )
ZMQ_CONSUMER_ADDR = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
    )

# Historical data path
AGENT_DATA_PATH = "agent_halide_data_uvvis_only.csv"
# AGENT_DATA_PATH = None

# Tiled URI for evaluation function
TILED_URI = os.environ.get(
    "TILED_URI", 
    "https://tiled.nsls2.bnl.gov"
    )
TILED_PROFILE = os.environ.get(
    "TILED_PROFILE", 
    "xpd"
    )

build_agent = BuildAgent(
    # False -- this notebook exercises build_local(), the no-queue-server
    # path (drives a local RunEngine directly instead of dispatching
    # acquisition plans by name through a Queue Server). build() (the
    # queue_server=True path) is what examples/halide_example_class.ipynb
    # demonstrates instead.
    queue_server = False,
    # Agent
    agent_data_path = AGENT_DATA_PATH,
    # Http 
    http_server_uri = HTTP_SERVER_URI,
    http_api_key = HTTP_API_KEY,
    # ZMQ
    zmq_consumer_address = ZMQ_CONSUMER_ADDR,
    # Tiled
    tiled_profile = TILED_PROFILE,
    # Evaluation Method
    evaluation_method = 'xray-uvvis'
)


# Provide the agent some metadata
build_agent.set_metadata({
    "beamline": "28id2",
    "tags": ["qserver", "bluesky"],
    "comment": "Halide Synthesis Test"
    })
print(build_agent.metadata_string)

/Users/work/Documents/BNL-bluesky/bluvenv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[INFO 09-17 13:49:12] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.51


beamline: 28id2
tags: ['qserver', 'bluesky']
comment: Halide Synthesis Test



## BEAMLINE - PDF Xray Diffraction

Objective correlation function with the phases that will be tested.


Correlation functions:
- pearson, nn-matrix, weighted-profile-r, cross-correlation, or ensemble

In [3]:
from xpd_tools.optimization.helpers.phases import Phase

# PDF correlation function
PDF_FUNCTION = "pearson"

# Set the phase selection objectives and directions
build_agent.set_xray_objectives(
    # Measurement
    max_retries     = 10,
    retry_delay     = 2.0,
    # Configuration
    exposure        = 600.0,
    frame_acq_time  = 1.5,
    no_dark         = False,
    stream_name     = "scattering",
    # Screening: UV-Vis is measured (screen_only/screen_and_record) or not
    # (unscreened). We configure full UV-Vis objectives below, so we want
    # the measurement recorded, not just used to gate acquisition.
    screening       = "screen_and_record",
    # Quality Checks
    use_good_bad    = True,
    good_target     = 2,
    max_bad         = 3,
    num_abs         = 16,
    num_flu         = 16,
    # PDF correlation masking window (Angstroms)
    min_radius      = 2.0,
    max_radius      = 20.0,
    # Phase Fitting
    objective_function = PDF_FUNCTION,
    phases   = [
        Phase(
            name      = "CsPbBr3",
            gr        = "refdata/CsPbBr3.gr",
            cif       = "refdata/CsPbBr3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsBr",
            gr        = "refdata/CsBr.gr",
            cif       = "refdata/CsBr.cif",
            minimize  = True
            ),
        Phase(
            name      = "Cs4PbBr6",
            gr        = "refdata/Cs4PbBr6.gr",
            cif       = "refdata/Cs4PbBr6.cif",
            minimize  = True
            ),
    ]
  )
print(build_agent.xray_settings)
print(build_agent.quality_policy)
print("screening:", build_agent.screening)
print("min/max radius:", build_agent.min_radius, build_agent.max_radius)

XraySettings(exposure=600.0, frame_acq_time=1.5, no_dark=False, stream_name='scattering')
QualityPolicy(enabled=True, good_batches=2, max_bad_batches=3, absorbance_shots=16, fluorescence_shots=16)
screening: screen_and_record
min/max radius: 2.0 20.0


## BEAMLINE - UVvis Screening

In [4]:
# Optimization Parameters
PEAK_TARGET         = 450   # nm
PEAK_TOLERANCE      = 5     # nm

from xpd_tools.optimization.helpers.qepro import PlqyReference, SpectraFitSettings

build_agent.set_uvvis_objectives(
    # Objective Target
    peak_target             = PEAK_TARGET,
    peak_tolerance          = PEAK_TOLERANCE,
    max_retries             = 10,
    retry_delay             = 2.0,
    # Screening, data selection, and fitting windows
    fit_settings = SpectraFitSettings(
        pl_screen_key_height        = 200,
        pl_screen_peak_height       = 50,
        pl_screen_peak_distance     = 100,
        pl_percent_range            = (40, 100),
        pl_wavelength_range         = (400, 800),
        pl_fit_maxfev               = 100000,
        pl_fit_r2_window_sigma      = 3,
        absorbance_percent_range    = (10, 70),
        absorbance_wavelength_range = (210, 700),
    ),
    # Calibration Standard Reference (reference_type defaults to "quinine")
    plqy = PlqyReference(
        excitation_wavelength_nm   = 365,
        absorbance                 = 0.361,
        pl_integral                = 952628,
        refractive_index           = 1.337,
        plqy                       = 0.546,
        solvent_refractive_index   = 1.506,
    ),
)

print(build_agent.fit_settings)
print(build_agent.plqy)

SpectraFitSettings(pl_screen_key_height=200, pl_screen_peak_height=50, pl_screen_peak_distance=100, pl_percent_range=(40, 100), pl_wavelength_range=(400, 800), pl_fit_maxfev=100000, pl_fit_r2_window_sigma=3, absorbance_percent_range=(10, 70), absorbance_wavelength_range=(210, 700))
PlqyReference(reference_type='quinine', excitation_wavelength_nm=365, absorbance=0.361, pl_integral=952628, refractive_index=1.337, plqy=0.546, solvent_refractive_index=1.506)


## BLOP - DOFs

Degrees of freedom that BLOP can modify

In [ ]:
# Set the Agent DOFs
from xpd_tools.optimization.helpers.dofs import Pump

build_agent.set_dofs(
    pumps = [
      Pump(
        name = "CsPb",
        bounds = (10, 200),
        id = "dds2_p1"
      ),
      Pump(
        name = "Br",
        bounds = (5, 200),
        id = "dds2_p2"
      ),
      Pump(
        name = "Cl",
        bounds = (0, 200),
        id = "dds3_p1"
      ),
      Pump(
        name = "OAm",
        bounds = (0, 200),
        id = "dds3_p2"
      )
    ]
  )
print(build_agent.dofs)

[RangeDOF(name='infusion_rate_CsPb', actuator=None, bounds=(10, 200), parameter_type='float', step_size=None, scaling=None), RangeDOF(name='infusion_rate_Br', actuator=None, bounds=(5, 200), parameter_type='float', step_size=None, scaling=None), RangeDOF(name='infusion_rate_Cl', actuator=None, bounds=(0, 200), parameter_type='float', step_size=None, scaling=None), RangeDOF(name='infusion_rate_OAm', actuator=None, bounds=(0, 200), parameter_type='float', step_size=None, scaling=None)]


## BEAMLINE - Experimental Harware Parameters

In [ ]:
from xpd_tools.optimization.plans import FlowSource, DilutionStage, WashCycle

build_agent.experiment(
    sources = [
        FlowSource(
            dof="infusion_rate_CsPb",
            pump='dds2_p1',
            precursor="CsPbOA",
            sample_label="CsPb",
        ),
        FlowSource(
            dof="infusion_rate_Br",
            pump='dds2_p2',
            precursor="TOABr",
            sample_label="Br",
        ),
        FlowSource(
            dof="infusion_rate_Cl",
            pump='dds3_p1',
            precursor="ZnCl2",
            sample_label="Cl",
        ),
        FlowSource(
            dof="infusion_rate_OAm",
            pump='dds3_p2',
            precursor="OAm_Tol",
            sample_label="OAm",
        ),
    ],
    dilutions = [
        DilutionStage(
            pump='dds1_p1',
            ratio=1.0,
            position="before_equilibrium",
            syringe_ml=20,
            material="plastic_BD",
            target_ml=20,
        ),
        DilutionStage(
            pump='ultra2',
            ratio=1.0,
            position="after_equilibrium",
            syringe_ml=100,
            material="steel",
            target_ml=100,
            wait_sec=30,
        ),
    ],
    wash_cycles=[
        WashCycle(pump='ultra1'),
    ]
)

ValueError: DOFs and flow sources must reference the same names: sources missing a DOF: ['infusion_rate_Cl2']; DOFs missing a source: ['infusion_rate_Cl']

## BLOP - Load and preprocess historical data

In [ ]:
if AGENT_DATA_PATH is not None:
    df = pd.read_csv(AGENT_DATA_PATH, index_col=0)
    df = df[["Peak", "log_FWHM", "log_PLQY", "infusion_rate_CsPb", "infusion_rate_Br", "infusion_rate_Cl"]]
    df["peak_distance"] = (df["Peak"] - PEAK_TARGET).abs()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    print(df)

           Peak  log_FWHM  log_PLQY  infusion_rate_CsPb  infusion_rate_Br  \
0    446.400000  2.918851 -0.671386           10.590000         31.760000   
1    463.410000  3.206398 -0.436956           12.650000         37.950000   
2    477.760000  3.087399 -0.016129           15.710000         47.130000   
3    492.630000  3.096934 -0.093212           17.870000         53.610000   
4    506.550000  3.069447 -0.406466           19.470000         58.410000   
..          ...       ...       ...                 ...               ...   
149  423.410474  2.797326 -3.357577           80.254345         54.826290   
150  476.235522  3.202355 -0.526928            5.000000        146.549312   
151  421.332148  2.616667 -2.630542           17.673281         39.181675   
152  433.281172  2.777166 -1.699943           28.232538         83.379452   
153  431.006425  2.738322 -2.020931           19.508029         58.222542   

     infusion_rate_Cl  peak_distance  
0           47.650000       3.600000

## BLOP - Build and Run Agent

In [ ]:
from bluesky.run_engine import RunEngine
from xpd_tools.optimization.legacy import (
    build_fake_tiled_clients,
    build_xpd_objects,
    identity_wrap_xray_run,
)

RE = RunEngine()

# Faked clients for offline runs
tiled_client, sandbox_client = build_fake_tiled_clients(RE, phases=build_agent.phases)

run_agent = build_agent.build_local(
    devices=build_xpd_objects(),
    wrap_xray_run=identity_wrap_xray_run,
    # Real hardware defaults (30cm mixer, ratio=1.0) compute a real
    # multi-minute equilibrium wait from the pump rate -- zero them out
    # for a fast local/simulated run.
    mixer_lengths_cm=(0.0,),
    residence_time_ratio=0.0,
    tiled_client=tiled_client,
    sandbox_client=sandbox_client,
)
run_agent.ax_client.configure_generation_strategy(
    initialize_with_center=False,
    # Loads historical data?
    use_existing_trials_for_initialization=True,
)


### BLOP - Run

In [ ]:
ITERATIONS = 3  # small on purpose -- this is for verifying the build/plan work, not a real campaign

RE(run_agent.optimize(iterations=ITERATIONS))

## Summarize and Export Data

In [ ]:
from pathlib import Path

Path("tmp/output").mkdir(parents=True, exist_ok=True)
df = run_agent.ax_client.summarize()
df.to_csv("tmp/output/agent_halide_data.csv")